---
## Task 3 — Discipline: Winning Teams vs. Losing Teams

### Analytic question formulation
Do **losing teams** pick up significantly more **cards (yellow + red combined)** per match
than **winning teams**? The idea being tested is that falling behind or being outplayed leads
teams to commit more fouls, and so accumulate more cards.

**H0:** mean total cards(losing team) = mean total cards(winning team)
**H1:** mean total cards(losing team) > mean total cards(winning team)

### Data wrangling
Load the raw match-level CSV, drop drawn matches (there is no "winner" to compare), combine
each team's yellow and red cards into a single `total_card` count, and reshape the data from
one row per match into one row per **team-match** observation flagged with whether that team
won (`win` = 1) or lost (`win` = 0).

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import scipy.stats as st

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your CSV file
file_path = '/content/drive/MyDrive/Colab/world_cup_match_data.csv'

df = pd.read_csv(file_path)
print("CSV imported successfully!")
print(df.head())

Mounted at /content/drive
CSV imported successfully!
    timestamp              date_GMT    status  attendance home_team_name  \
0  1781204400  Jun 11 2026 - 7:00pm  complete       80824         Mexico   
1  1781229600  Jun 12 2026 - 2:00am  complete       44985    South Korea   
2  1781290800  Jun 12 2026 - 7:00pm  complete       43002         Canada   
3  1781312400  Jun 13 2026 - 1:00am  complete       70492          USMNT   
4  1781377200  Jun 13 2026 - 7:00pm  complete       67966          Qatar   

           away_team_name  referee  Game Week  Pre-Match PPG (Home)  \
0            South Africa      NaN        1.0                   0.0   
1          Czech Republic      NaN        1.0                   0.0   
2  Bosnia and Herzegovina      NaN        1.0                   0.0   
3                Paraguay      NaN        1.0                   0.0   
4             Switzerland      NaN        1.0                   0.0   

   Pre-Match PPG (Away)  ...  odds_ft_home_team_win  odds_ft_dr

In [2]:
df3 = df[
        [
            'home_team_name',
            'away_team_name',
            'home_team_goal_count',
            'away_team_goal_count',
            'home_team_yellow_cards',
            'away_team_yellow_cards',
            'home_team_red_cards',
            'away_team_red_cards'
        ]
    ].copy()

# Determine winning team
# 0 = Home team wins, 1 = Away team wins, -1 = Draw
df3['winning_team'] = df3.apply(
    lambda row: 0 if row['home_team_goal_count'] > row['away_team_goal_count']
    else 1 if row['away_team_goal_count'] > row['home_team_goal_count']
    else -1,
    axis=1
)
df3 = df3[df3['winning_team'] != -1].copy()

df3['home_team_total_card'] = df3['home_team_yellow_cards'] + df3['home_team_red_cards']
df3['away_team_total_card'] = df3['away_team_yellow_cards'] + df3['away_team_red_cards']
df3['match'] = df3['home_team_name'] + ' vs ' + df3['away_team_name']

df3_long = pd.concat([
    df3[['match', 'home_team_name', 'home_team_total_card', 'winning_team']]
        .rename(columns={'home_team_name': 'team', 'home_team_total_card': 'total_card'})
        .assign(win=lambda x: (x['winning_team'] == 0).astype(int)),

    df3[['match', 'away_team_name', 'away_team_total_card', 'winning_team']]
        .rename(columns={'away_team_name': 'team', 'away_team_total_card': 'total_card'})
        .assign(win=lambda x: (x['winning_team'] == 1).astype(int))
], ignore_index=True)

df3_long = df3_long.drop(columns='winning_team')
print("Population size (team-match observations):", len(df3_long))
df3_long.head()

Population size (team-match observations): 160


,match,team,total_card,win
0,Mexico vs South Africa,Mexico,2,1
1,South Korea vs Czech Republic,South Korea,1,1
2,USMNT vs Paraguay,USMNT,1,1
3,Haiti vs Scotland,Haiti,1,0
4,Australia vs Turkey,Australia,0,1


### Data preparation and sampling
**Population:** every team-match observation, split into a winning-team group (`win` = 1) and
a losing-team group (`win` = 0), each carrying a `total_card` count.
**Sample:** an independent **simple random sample of n = 30** drawn separately from the
winning-team cards and the losing-team cards, for a two-sample comparison.

In [3]:
winning_team_cards = df3_long[df3_long['win'] == 1]['total_card']
losing_team_cards  = df3_long[df3_long['win'] == 0]['total_card']

print("Population size — winning teams:", len(winning_team_cards))
print("Population size — losing teams:", len(losing_team_cards))

n = 30
win_sample  = winning_team_cards.sample(n=n, random_state=7).reset_index(drop=True)
lose_sample = losing_team_cards.sample(n=n, random_state=7).reset_index(drop=True)

print("Winners sample n:", len(win_sample), " Losers sample n:", len(lose_sample))

Population size — winning teams: 80
Population size — losing teams: 80
Winners sample n: 30  Losers sample n: 30


### Descriptive statistics

In [4]:
for name, s in [("Winning teams", win_sample), ("Losing teams", lose_sample)]:
    print(f"\n{name}:")
    print(s.describe())


Winning teams:
count    30.000000
mean      0.866667
std       0.937102
min       0.000000
25%       0.000000
50%       1.000000
75%       1.000000
max       3.000000
Name: total_card, dtype: float64

Losing teams:
count    30.000000
mean      1.766667
std       1.501340
min       0.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       6.000000
Name: total_card, dtype: float64


### Inferential statistics — Confidence intervals (95%, each group)

In [5]:
for name, s in [("Winning teams", win_sample), ("Losing teams", lose_sample)]:
    m, sem = s.mean(), st.sem(s)
    ci = st.t.interval(0.95, df=len(s) - 1, loc=m, scale=sem)
    print(f"{name}: mean = {m:.3f}, 95% CI = ({ci[0]:.3f}, {ci[1]:.3f})")

Winning teams: mean = 0.867, 95% CI = (0.517, 1.217)
Losing teams: mean = 1.767, 95% CI = (1.206, 2.327)


### Inferential statistics — Two-sample t-Test (Welch's, one-tailed)
H0: μ(lose) = μ(win)  vs.  H1: μ(lose) > μ(win)

In [6]:
t_stat, p_val = st.ttest_ind(lose_sample, win_sample, equal_var=False, alternative='greater')
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")

alpha = 0.05
conclusion = "Reject H0" if p_val < alpha else "Fail to reject H0"
print("Conclusion:", conclusion, "at the 5% significance level.")

t-statistic = 2.785, p-value = 0.0038
Conclusion: Reject H0 at the 5% significance level.


In [7]:
# diff = lose_sample.mean() - win_sample.mean()
# sig = "statistically significant" if p_val < alpha else "not statistically significant"
# print(
#     f"Interpretation: In the sample, losing teams averaged {lose_sample.mean():.2f} cards "
#     f"vs {win_sample.mean():.2f} for winning teams (difference = {diff:.2f}). "
#     f"With p = {p_val:.4f}, this result is {sig} at the 5% level, so we {conclusion.lower()} "
#     f"that losing teams pick up significantly more cards than winning teams."
# )

Interpretation: In the sample, losing teams averaged 1.77 cards vs 0.87 for winning teams (difference = 0.90). With p = 0.0038, this result is statistically significant at the 5% level, so we reject h0 that losing teams pick up significantly more cards than winning teams.


**Interpretation:** losing teams pick up significantly more cards than winning teams.